# TactiVision: Fine-tune โมเดลฟุตบอลเดิม
เลือก Runtime → Change runtime type → T4 GPU แล้วรันจากบนลงล่าง

อัปโหลด **football-v1-ready.zip** ที่จัดชุดแล้ว ไม่ใช่ ZIP เดิมจาก Roboflow

Train: 37 ภาพจาก PFYsHN24C3k / Validation: 36 ภาพจาก tJnScrAKJfI เป็นคนละแมตช์กัน ไม่มีภาพ Test ใน Notebook นี้

รอบนี้ Fine-tune เฉพาะ `football-player-detection.pt` ซึ่งตรวจ 4 คลาส ไม่ได้ Fine-tune โมเดลตรวจบอลแยก และยังไม่รวม Dataset สาธารณะเพิ่มเติม

คะแนนท้าย Notebook เป็น Validation เพื่อเลือกโมเดล ต้องวัด Test จากแมตช์ idq95Xp5RI0 อีกครั้งหลังเลือกโมเดลและค่าตั้งเสร็จแล้ว

ข้อมูล Train ยังมีเพียงหนึ่งแมตช์ จึงเป็นการทดลองรอบแรก ผลอาจดีขึ้นหรือแย่ลงได้

In [ ]:
%pip install -q 'ultralytics>=8.3,<9' gdown pyyaml pillow


In [ ]:
from pathlib import Path
from datetime import datetime
import gc, json, shutil, zipfile
import torch, yaml, gdown
from ultralytics import YOLO
from google.colab import files
from PIL import Image as PILImage, ImageDraw
from IPython.display import display

assert torch.cuda.is_available(), 'เลือก Runtime → Change runtime type → T4 GPU ก่อน'
print('GPU:', torch.cuda.get_device_name(0))
RUN_ROOT = Path('/content/tactivision_runs') / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_ROOT.mkdir(parents=True, exist_ok=False)
print('โฟลเดอร์รอบนี้:', RUN_ROOT)


## อัปโหลด football-v1-ready.zip
เลือกไฟล์ ZIP ที่จัดชุดแล้วเพียงไฟล์เดียว

In [ ]:
uploaded = files.upload()
zip_paths = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
assert len(zip_paths) == 1, 'อัปโหลด football-v1-ready.zip ไฟล์เดียว'
DATA_ROOT = RUN_ROOT / 'dataset'
DATA_ROOT.mkdir()
with zipfile.ZipFile(zip_paths[0]) as archive:
    for member in archive.infolist():
        destination = (DATA_ROOT / member.filename).resolve()
        assert destination.is_relative_to(DATA_ROOT.resolve()), 'ZIP มีพาธที่ไม่ถูกต้อง'
    archive.extractall(DATA_ROOT)
DATA_YAML = DATA_ROOT / 'data.yaml'
config = yaml.safe_load(DATA_YAML.read_text())
EXPECTED = ['ball', 'goalkeeper', 'player', 'referee']
assert config['names'] == EXPECTED, 'ชื่อหรือลำดับคลาสไม่ตรงกับโมเดลเดิม'
assert (DATA_ROOT / 'audit.json').is_file(), 'ใช้ ZIP ที่จัดชุดแล้ว ไม่ใช่ ZIP เดิมจาก Roboflow'
audit = json.loads((DATA_ROOT / 'audit.json').read_text())
assert audit['splits']['train']['images'] == 37 and audit['splits']['valid']['images'] == 36
assert audit['removed_duplicate_boxes'] == 13
assert not config.get('test'), 'Notebook รอบนี้ไม่ใช้ Test'
for split, expected_count in [('train', 37), ('valid', 36)]:
    images = sorted((DATA_ROOT / split / 'images').glob('*.jpg'))
    assert len(images) == expected_count, f'{split}: จำนวนภาพผิด'
    for image in images:
        assert 'idq95Xp5RI0' not in image.name
        assert (DATA_ROOT / split / 'labels' / (image.stem + '.txt')).is_file()
    print(split, len(images), 'ภาพ')
config['path'] = str(DATA_ROOT)
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
del uploaded


## ดาวน์โหลดโมเดลฟุตบอลเดิม
ไฟล์อ้างอิงจาก setup.sh ของ https://github.com/roboflow/sports/tree/main/examples/soccer

In [ ]:
BASE_MODEL = Path('/content/football-player-detection.pt')
if not BASE_MODEL.is_file():
    downloaded = gdown.download(id='17PXFNlx-jI7VjVo_vQnB1sONjRyvoB-q', output=str(BASE_MODEL), quiet=False)
    assert downloaded and BASE_MODEL.is_file(), 'ดาวน์โหลดโมเดลไม่สำเร็จ'
baseline = YOLO(str(BASE_MODEL))
assert [baseline.names[i] for i in range(len(baseline.names))] == EXPECTED, baseline.names
print('โมเดลเดิม:', baseline.names)
IMGSZ = 1280
BATCH = 1  # โมเดลต้นฉบับค่อนข้างใหญ่ เริ่มทีละภาพบน T4
EVAL_ARGS = dict(data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH, device=0,
                 conf=0.001, iou=0.7, workers=2, plots=True, project=str(RUN_ROOT))


## วัดโมเดลเดิมบน Validation
เก็บคะแนนก่อนเรียนเพิ่ม เพื่อเปรียบเทียบด้วยภาพและค่าตั้งเดียวกัน

In [ ]:
def metric_summary(metrics):
    return {'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
            'mAP50': float(metrics.box.map50), 'mAP50_95': float(metrics.box.map),
            'per_class_mAP50_95': {name: float(value) for name, value in zip(EXPECTED, metrics.box.maps)}}
before_metrics = baseline.val(name='baseline_validation', **EVAL_ARGS)
before = metric_summary(before_metrics)
print(json.dumps(before, indent=2))
del baseline, before_metrics
gc.collect()
torch.cuda.empty_cache()


## Fine-tune
เริ่มใหม่จากโมเดลฟุตบอลเดิม แล้วเรียนเพิ่มสูงสุด 40 Epoch เลือก best.pt ด้วย Validation

หากเกิด CUDA out of memory ทั้งที่ batch=1 ให้เปลี่ยน IMGSZ เป็น 960 ในเซลล์ตั้งค่า แล้วรันวัดโมเดลเดิมและเทรนใหม่ด้วยขนาดเดียวกัน

รอบนี้ใช้ learning rate ต่ำเป็นค่าเริ่มต้น ยังไม่ได้พิสูจน์ว่าเป็นค่าที่ดีที่สุดสำหรับข้อมูลนี้

In [ ]:
model = YOLO(str(BASE_MODEL))
trained = model.train(
    data=str(DATA_YAML), epochs=40, patience=10, imgsz=IMGSZ, batch=BATCH, device=0,
    optimizer='AdamW', lr0=0.0001, seed=42, deterministic=True, workers=2, cache=False,
    project=str(RUN_ROOT), name='finetune', plots=True
)
TRAIN_DIR = Path(trained.save_dir)
BEST_MODEL = TRAIN_DIR / 'weights' / 'best.pt'
assert BEST_MODEL.is_file(), 'ยังไม่มี best.pt ต้องรอเทรนให้จบ'
print('โมเดลหลัง Fine-tune:', BEST_MODEL)
del model, trained
gc.collect()
torch.cuda.empty_cache()


## เปรียบเทียบคะแนนบน Validation
คะแนนนี้ยังไม่ใช่ Test และ Fine-tune อาจไม่ได้ทำให้ดีขึ้นทุกคลาส

In [ ]:
finetuned = YOLO(str(BEST_MODEL))
after_metrics = finetuned.val(name='finetuned_validation', **EVAL_ARGS)
after = metric_summary(after_metrics)
import pandas as pd
table = pd.DataFrame({'ก่อน Fine-tune': {k:v for k,v in before.items() if isinstance(v, float)},
                      'หลัง Fine-tune': {k:v for k,v in after.items() if isinstance(v, float)}})
display(table.round(4))
display(pd.DataFrame({'ก่อน Fine-tune': before['per_class_mAP50_95'],
                      'หลัง Fine-tune': after['per_class_mAP50_95']}).round(4))
comparison = {'split': 'validation_not_test', 'imgsz': IMGSZ, 'eval_conf': 0.001, 'eval_iou': 0.7,
              'before': before, 'after': after}
(RUN_ROOT / 'validation_comparison.json').write_text(json.dumps(comparison, indent=2))
del finetuned, after_metrics
gc.collect()
torch.cuda.empty_cache()


## ดูภาพเทียบกัน: ซ้ายเดิม / ขวาหลัง Fine-tune
ใช้ Confidence 0.15 เท่ากันทั้งสองฝั่ง แสดงตัวอย่าง 4 ภาพ และบันทึกคู่เทียบครบ 36 ภาพใน ZIP ท้าย Notebook

ค่าที่ใช้วาดภาพต่างจากค่า conf ของการวัด mAP เพราะ mAP ต้องดูผลหลายระดับความมั่นใจ

In [ ]:
PAIR_DIR = RUN_ROOT / 'side_by_side_validation'
PAIR_DIR.mkdir(exist_ok=True)
validation_images = sorted((DATA_ROOT / 'valid' / 'images').glob('*.jpg'))
old_model = YOLO(str(BASE_MODEL)).to('cpu')
new_model = YOLO(str(BEST_MODEL)).to('cpu')
for index, image_path in enumerate(validation_images):
    panels = []
    for detector in (old_model, new_model):
        result = detector.predict(str(image_path), imgsz=IMGSZ, conf=0.15, iou=0.7, device=0, save=False, verbose=False)[0]
        panels.append(PILImage.fromarray(result.plot(line_width=1)[:, :, ::-1]))
        detector.to('cpu')
        torch.cuda.empty_cache()
    width, height = panels[0].size
    pair = PILImage.new('RGB', (width*2, height+35), 'white')
    pair.paste(panels[0], (0,35))
    pair.paste(panels[1], (width,35))
    draw = ImageDraw.Draw(pair)
    draw.text((10,10), 'BEFORE - original model', fill='black')
    draw.text((width+10,10), 'AFTER - fine-tuned model', fill='black')
    pair.save(PAIR_DIR / image_path.name, quality=95)
    if index < 4:
        print(image_path.name.split('_jpg')[0])
        display(pair.resize((1280, round(pair.height*1280/pair.width))))
print('บันทึกภาพเปรียบเทียบ:', len(validation_images), 'คู่')
del old_model, new_model
gc.collect()
torch.cuda.empty_cache()


## ดาวน์โหลดโมเดลและผลลัพธ์
ZIP มี best.pt, data.yaml, คะแนนก่อน/หลัง, รายการแบ่งภาพ, กราฟ และภาพเทียบกัน ดาวน์โหลดเก็บไว้ทันทีเพราะไฟล์ใน Colab เป็นไฟล์ชั่วคราว

In [ ]:
BUNDLE = RUN_ROOT / 'deliverables'
BUNDLE.mkdir(exist_ok=True)
shutil.copy2(BEST_MODEL, BUNDLE / 'best.pt')
shutil.copy2(DATA_ROOT / 'data.yaml', BUNDLE / 'data.yaml')
for name in ('audit.json', 'split_manifest.json'):
    shutil.copy2(DATA_ROOT / name, BUNDLE / name)
shutil.copy2(RUN_ROOT / 'validation_comparison.json', BUNDLE / 'validation_comparison.json')
shutil.copytree(PAIR_DIR, BUNDLE / 'side_by_side_validation', dirs_exist_ok=True)
for folder in (TRAIN_DIR, RUN_ROOT / 'baseline_validation', RUN_ROOT / 'finetuned_validation'):
    destination = BUNDLE / folder.name
    destination.mkdir(exist_ok=True)
    for artifact in folder.iterdir():
        if artifact.is_file() and artifact.suffix in {'.png', '.jpg', '.csv', '.yaml'}:
            shutil.copy2(artifact, destination / artifact.name)
from importlib.metadata import version
(BUNDLE / 'environment.json').write_text(json.dumps({p:version(p) for p in ('ultralytics','torch','torchvision','numpy','pillow')}, indent=2))
bundle_zip = shutil.make_archive(str(RUN_ROOT / 'tactivision_finetune_results'), 'zip', BUNDLE)
print('ดาวน์โหลดเสร็จแล้วให้เก็บไฟล์นี้ไว้:', bundle_zip)
files.download(bundle_zip)
